<a href="https://colab.research.google.com/github/DiploDatos/AprendizajePorRefuerzos/blob/master/lab_1_intro_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Introducción al aprendizaje por refuerzos

Curso Aprendizaje por Refuerzos, Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones

Facultad de Matemática, Astronomía, Física y Computación, Universidad Nacional de Córdoba

## Introducción

Este laboratorio usa CliffWalking para comparar SARSA y Q-Learning. Las funciones de actualización son ejercicios: antes de ejecutar el entrenamiento, completá una de ellas y seleccionála en la celda de configuración. Las visualizaciones muestran el aprendizaje de cada corrida y los valores finales de la tabla Q.

### Repaso rápido


* **Recompensa:** señal $r$ que recibe el agente después de cada acción.

* **Valor:** $V_\pi(s)$ es la recompensa futura esperada desde $s$ al seguir $\pi$; $Q_\pi(s,a)$ es la recompensa futura esperada al tomar $a$ en $s$ y continuar con $\pi$.

* **Política:** indica cómo elegir acciones. En una política $\epsilon$-greedy, con probabilidad $1-\epsilon$ se elige una acción con el mayor valor Q y con probabilidad $\epsilon$ se elige una acción al azar. Si hay una única acción óptima entre $|A|$ acciones, su probabilidad total es $1-\epsilon+\epsilon/|A|$; la de cada otra acción es $\epsilon/|A|$.

* **Softmax:** asigna una probabilidad a cada acción según su valor y luego realiza un sorteo: $\pi(a\mid s)=\frac{e^{Q(s,a)/\tau}}{\sum_{b\in A}e^{Q(s,b)/\tau}}$. El parámetro $\tau>0$ controla cuán concentrada está la elección en las acciones más valiosas.

En este notebook vemos dos algoritmos para actualizar la función de valor (y, por lo tanto, la política de selección de acciones):

* Actualización por SARSA (on-policy).

$$Q(s,a) \gets Q(s,a) + \alpha (r + \gamma Q(s',a') - Q(s,a))$$

Algoritmo completo (a modo de referencia):

![Algoritmo SARSA](https://github.com/DiploDatos/AprendizajePorRefuerzos/blob/master/images/sarsa.png?raw=1)


* Actualización por Q-Learning (*off-policy*):

$$Q(s,a) \gets Q(s,a) + \alpha\left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]$$

Si $s'$ es terminal, el valor futuro del objetivo es cero.

Algoritmo completo (a modo de referencia):

![Algoritmo Q-Learning](https://github.com/DiploDatos/AprendizajePorRefuerzos/blob/master/images/q_learning.png?raw=1)

Fuente de las imágenes: capítulo 6 de [Reinforcement Learning: An Introduction](http://www.incompleteideas.net/book/the-book.html).

## Librería a usar: Gymnasium (sucesora de OpenAI Gym)

[Gymnasium](https://github.com/Farama-Foundation/Gymnasium), es la librería que continúa con [OpenAI Gym](https://gym.openai.com/) (Brockman et. al., 2016), famosa librería de OpenAI que ofrece entornos y una interfaz estándar con la cuál probar nuestros agentes. Su objetivo es proveer benchmarks unificados para ver el desempeño de algoritmos en el entorno y así poder saber con facilidad cómo es su desempeño comparado con los demás. Parte de la siguiente sección está basada en la [documentación oficial de OpenAI](https://gym.openai.com/docs/).

La interfaz principal de Gymnasium posee cuatro métodos:

* `reset(seed=...)`: reinicia el entorno y devuelve `(observación, info)`.
* `step(action)`: avanza un paso y devuelve `(observación, recompensa, terminated, truncated, info)`.
* `render()`: produce una imagen o muestra el entorno según el `render_mode` indicado al crearlo.
* `close()`: libera los recursos del entorno.

Los atributos `action_space`, `observation_space` y `reward_range` describen las acciones, observaciones y recompensas posibles.

Los dos ejemplos siguientes muestran una imagen del entorno y ejecutan unas pocas acciones aleatorias. Para esos ejemplos se usa `render_mode="rgb_array"`; no hace falta instalar `ffmpeg`. Para una ventana interactiva local, se puede crear un entorno aparte con `render_mode="human"`.

En Colab se instalan las dependencias en la celda siguiente. Para ejecutarlo localmente con Python 3.12, se pueden instalar las dependencias del curso con `python -m pip install -r 2026/huggingface/requirements.txt` desde la raíz del repositorio.

Código básico de importación y funciones de graficación (no modificar)

In [ ]:
# @title Gráficos de entrenamiento y tabla Q
import numpy as np
import matplotlib.pyplot as plt


def _plot_series(values, title, ylabel, window=50):
    values = np.asarray(values, dtype=float)
    episodes = np.arange(1, len(values) + 1)
    plt.figure(figsize=(9, 4))
    plt.plot(episodes, values, alpha=0.3, label="Por episodio")
    if len(values):
        width = min(window, len(values))
        smooth = np.convolve(values, np.ones(width) / width, mode="valid")
        plt.plot(episodes[width - 1:], smooth, label=f"Media móvil ({width})")
    plt.xlabel("Episodio")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.show()


def plot_reward_per_episode(reward_ep):
    _plot_series(reward_ep, "Recompensa durante el entrenamiento", "Recompensa")


def plot_steps_per_episode(timesteps_ep):
    _plot_series(timesteps_ep, "Pasos durante el entrenamiento", "Pasos")


def draw_value_matrix(q):
    rows, cols = 4, 12
    values = np.full((rows, cols), np.nan)
    arrows = ("↑", "→", "↓", "←")
    labels = np.full((rows, cols), "?", dtype=object)

    for row in range(rows):
        for col in range(cols):
            state = row * cols + col
            if row == 3 and 1 <= col <= 10:
                labels[row, col] = "C"
            elif state == 47:
                labels[row, col] = "G"
            elif any((state, action) in q for action in range(4)):
                action_values = [q.get((state, action), 0.0) for action in range(4)]
                values[row, col] = max(action_values)
                prefix = "S " if state == 36 else ""
                labels[row, col] = f"{prefix}{arrows[np.argmax(action_values)]}\n{values[row, col]:.1f}"
            elif state == 36:
                labels[row, col] = "S ?"

    cmap = plt.colormaps["RdYlGn"].copy()
    cmap.set_bad("#eeeeee")
    fig, ax = plt.subplots(figsize=(14, 5))
    image = ax.imshow(np.ma.masked_invalid(values), cmap=cmap)
    for row in range(rows):
        for col in range(cols):
            ax.text(col, row, labels[row, col], ha="center", va="center", fontsize=9)
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_title("Mejor valor Q y acción por estado; ? = sin visitar, C = acantilado")
    fig.colorbar(image, ax=ax, label="máx Q(s, a)")
    fig.tight_layout()
    plt.show()

Ejemplo: agente CartPole

In [ ]:
import importlib.util

if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    %pip install -q "gymnasium[classic-control]>=1.2,<1.5" numpy matplotlib

import gymnasium as gym

env_demo = gym.make("CartPole-v1", render_mode="rgb_array")
observation, info = env_demo.reset(seed=42)
for _ in range(20):
    observation, reward, terminated, truncated, info = env_demo.step(
        env_demo.action_space.sample()
    )
    if terminated or truncated:
        observation, info = env_demo.reset()
plt.imshow(env_demo.render())
plt.axis("off")
plt.show()
env_demo.close()

Ejemplo: agente Mountain Car

In [ ]:
env_demo = gym.make("MountainCar-v0", render_mode="rgb_array")
observation, info = env_demo.reset(seed=42)
for _ in range(20):
    observation, reward, terminated, truncated, info = env_demo.step(
        env_demo.action_space.sample()
    )
    if terminated or truncated:
        observation, info = env_demo.reset()
plt.imshow(env_demo.render())
plt.axis("off")
plt.show()
env_demo.close()

## Ejemplo 1: The Cliff.


![](https://github.com/GIDISIA/RLDiplodatos/blob/master/images/cliffwalking.png?raw=1)

donde S= starting point, G= goal

(imagen de Sutton y Barto, 2018)

Descripción del entorno:

Acciones:

* $\uparrow$ - Arriba
* $\downarrow$ - Abajo
* $\rightarrow$ - Derecha
* $\leftarrow$ - Izquierda

Función de recompensa:

* $-1$ en todos los demás estados
* $-100$ en el acantilado

Nota: caer en el acantilado devuelve al agente al estado inicial en un mismo episodio

Vemos los bloques básicos de nuestro agente

Definimos el método de elección de acciones. En este caso el mismo utiliza la política de exploración $\epsilon$-greedy.

In [ ]:
def choose_action_e_greedy(
        state: int,
        actions: range,
        q: dict,
        hyperparameters: dict,
        random_state: np.random.RandomState,
) -> int:
    """
    Elije una acción de acuerdo al aprendizaje realizado previamente
    usando una política de exploración épsilon-greedy
    """
    # ej: para 4 acciones inicializa en [0,0,0,0]
    q_values = [q.get((state, a), 0.0) for a in actions]
    max_q = max(q_values)
    # sorteamos un número: es menor a épsilon?
    if random_state.uniform() < hyperparameters['epsilon']:
        # sí: se selecciona una acción aleatoria
        return random_state.choice(actions)

    count = q_values.count(max_q)

    # hay más de un máximo valor de estado-acción?
    if count > 1:
        # sí: seleccionamos uno de ellos aleatoriamente
        best = [i for i in range(len(actions)) if q_values[i] == max_q]
        i = random_state.choice(best)
    else:
        # no: seleccionamos el máximo valor de estado-acción
        i = q_values.index(max_q)

    return actions[i]

In [ ]:
def choose_action_softmax(state, actions, q, hyperparameters, random_state):
    """Ejercicio optativo: sortear una acción según softmax(Q(s, a) / tau)."""
    raise NotImplementedError("Completar la política Softmax (actividad optativa)")

Las dos funciones siguientes son ejercicios. Ambas reciben `terminated` para que el valor futuro sea cero cuando se llega a la meta. La función Q-Learning conserva `next_action` en la firma para poder usar el mismo bucle de entrenamiento; no necesita ese argumento para calcular su objetivo.

In [ ]:
def learn_SARSA(state, action, reward, next_state, next_action,
                terminated, hyperparameters, q):
    """Actualizar q[(state, action)] con el objetivo SARSA.

    Si terminated es True, el objetivo es solamente reward.
    En otro caso, usar q.get((next_state, next_action), 0.0).
    """
    raise NotImplementedError("Completá learn_SARSA antes de entrenar")

In [ ]:
def learn_Q_learning(state, action, reward, next_state, next_action,
                     terminated, hyperparameters, q):
    """Actualizar q[(state, action)] con el objetivo Q-Learning.

    Si terminated es True, el objetivo es solamente reward.
    En otro caso, usar el máximo Q de las acciones posibles en next_state.
    next_action se recibe por compatibilidad con run(), pero no se utiliza.
    """
    raise NotImplementedError("Completá learn_Q_learning antes de entrenar")

Finalmente, definimos el método principal de iteraciones.

In [ ]:
def run(learning_function, hyperparameters, episodes_to_run, env, actions,
        q, random_state, max_steps=2000, choose_action=choose_action_e_greedy):
    """Entrena un agente en CliffWalking y registra pasos y recompensa reales."""
    timesteps_of_episode = []
    reward_of_episode = []

    for episode in range(episodes_to_run):
        state, _ = env.reset(seed=episode)
        action = choose_action(
            state, actions, q, hyperparameters, random_state
        )
        episode_reward = 0.0

        for step in range(max_steps):
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            next_action = None if terminated else choose_action(
                next_state, actions, q, hyperparameters, random_state
            )
            learning_function(
                state, action, reward, next_state, next_action,
                terminated, hyperparameters, q
            )
            if terminated or truncated:
                break
            state, action = next_state, next_action

        timesteps_of_episode.append(step + 1)
        reward_of_episode.append(episode_reward)

    steps = np.asarray(timesteps_of_episode)
    rewards = np.asarray(reward_of_episode)
    return steps.mean(), steps, rewards

Configurá el experimento. Antes de cada comparación, ejecutá de nuevo esta celda: reinicia la tabla Q y el generador aleatorio. Elegí `learn_SARSA` o `learn_Q_learning` después de completar la función correspondiente.

In [ ]:
q = {}  # tabla nueva para cada corrida
hyperparameters = {
    "alpha": 0.5,
    "gamma": 1.0,
    "epsilon": 0.1,
    "tau": 25,  # para la actividad optativa Softmax
}

learning_function = learn_SARSA  # cambiar por learn_Q_learning si corresponde
action_policy = choose_action_e_greedy  # Softmax es optativa
episodes_to_run = 500

env = gym.make("CliffWalking-v1", is_slippery=False)
actions = range(env.action_space.n)
random_state = np.random.RandomState(42)

Entrená el agente. Si aparece `NotImplementedError`, completá la función elegida antes de continuar con los gráficos.

In [ ]:
avg_steps_per_episode, timesteps_ep, reward_ep = run(
    learning_function, hyperparameters, episodes_to_run,
    env, actions, q, random_state, choose_action=action_policy
)
print(f"Pasos medios por episodio: {avg_steps_per_episode:.1f}")
print(f"Recompensa media por episodio: {reward_ep.mean():.1f}")

### Análisis de la ejecución del agente

#### Análisis de convergencia

A diferencia de lo que sucede en el aprendizaje supervisado, en el aprendizaje por refuerzos el rendimiento se evalúa por una función específica que es la función de recompensa. En la práctica, la función de recompensa puede ser externa (y provista por el entorno) o bien puede ser una función creada por diseño (a modo de dirigir el agente hacia lo que por diseño se considera mejor, en nuestro ejemplo podría ser con una recompensa de $+1$ cada vez que el agente llega al estado objetivo). Esto se conoce como *reward shaping*, y hay que tener mucho cuidado con los posibles efectos secundarios de su uso.

Como el objetivo de RL es maximizar la recompensa obtenida, es posible utilizar la información sobre la obtención de la recompensas en cada time-step o episodio para evaluar el rendimiento parcial del agente (esto depende mucho de la particularidad de la distribución de la recompensa para el problema tratado).

La recompensa y los pasos se muestran por episodio. La línea suavizada es una **media móvil de 50 episodios**. En CliffWalking, una recompensa más alta suele significar menos pasos o menos caídas al acantilado. Estas curvas corresponden al entrenamiento con exploración ε-greedy.

Veamos la recompensa real de cada episodio. Cada paso cuesta −1 y caer al acantilado cuesta −100.

In [ ]:
plot_reward_per_episode(reward_ep)

Veamos los pasos por episodio, con la misma media móvil.

In [ ]:
plot_steps_per_episode(timesteps_ep)

Para comparar SARSA y Q-Learning, guardá las curvas obtenidas, cambiá `learning_function` y ejecutá otra vez desde la celda de configuración. Ambas corridas deben empezar con una tabla Q vacía.

#### Análisis de la tabla Q

Cada casilla visitada muestra $\max_a Q(s,a)$ y una de las acciones con mayor valor. `?` indica un estado sin visitar, `C` el acantilado, `S` el inicio y `G` la meta. Los estados terminales no tienen una acción siguiente. Si hay empate entre acciones, la flecha mostrada es solamente una de las posibles.

Este mapa permite estudiar la política en un entorno tabular pequeño. Para evaluar la política final, además conviene ejecutar episodios de prueba sin exploración y medir su recompensa.

In [ ]:
draw_value_matrix(q)

In [ ]:
env.close()

## Actividades

1. Implementar y ejecutar SARSA en CliffWalking.
2. Implementar y ejecutar Q-Learning en CliffWalking. Comparar las curvas obtenidas con tablas Q nuevas y la misma semilla. ¿Por qué pueden diferir?
3. Cambiar $\alpha$, $\epsilon$ y $\gamma$ de a uno por vez. Describir su efecto sobre el entrenamiento.
4. (Opcional) Implementar la política de exploración Softmax, elegirla en `action_policy` y compararla con ε-greedy.
5. (Opcional) Implementar Dyna-Q a partir de Q-Learning y comparar su desempeño.

Podés guardar una copia del notebook resuelto en tu repositorio para conservar tus resultados.

FIN